# Abstract and Discussion statistics

The Abstract (v28d line 35) and the Discussion (v28d lines 181-193) print a few numbers that no figure panel shows: the cross-fitted peptide gains over matched metagenome databases, the count of selected databases below the human reference proteome, the reservoir sizes and the computing times. This notebook recomputes each of them from tables in `source_data/` that other units already ship (`Fig1`, `Fig2`, `Fig4`, `SFig15`). There is no figure panel and therefore no pixel check.

Run from the repository root: `python paper.py build abstract_discussion`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name   # holds only README.md: every table lives with the unit that first needs it
SD_FIG1 = REPO / "source_data" / "Fig1"
SD_FIG2 = REPO / "source_data" / "Fig2"
SD_FIG4 = REPO / "source_data" / "Fig4"
SD_SFIG15 = REPO / "source_data" / "SFig15"
import numpy, pandas, platform
import stats
print("python", platform.python_version(), "| numpy", numpy.__version__, "| pandas", pandas.__version__)

python 3.12.4 | numpy 2.2.6 | pandas 2.3.3


## Statistic: cross-fitted gain over matched metagenome databases

Claim (v28d line 35): "FastaLake identified 3.5–5.5% fewer peptides than matched metagenomic databases in two stool cohorts and 21–36% more in intestinal fluid". Discussion line 183 repeats it in words ("slightly fewer 9–50-residue peptides ... in stool and substantially more in intestinal fluid").

Per acquisition, gain = 100 × (FL / MG − 1), where FL and MG are peptides of 9–50 residues accepted at pooled peptide q ≤ 0.01 after 10-fold cross-fitting (source_data/Fig4/crossfit_per_acquisition.tsv, columns `FL_pooled_9`, `MG_pooled_9`). The printed values are medians over acquisitions: 3.5% is PREDICT (296 acquisitions, 97 participant clusters), 5.5% is CapScan stool (56 acquisitions, 15 participants), and 21–36% is the range of the medians of the four intestinal sampling regions (51–65 acquisitions each, 13–14 participants). Intervals: 95% percentile bootstrap over participants, 10,000 draws, seed 20261004, reproduced exactly as the original code drew them (two random streams, strata in fixed order). No hypothesis test is made.

In [2]:
cf = stats.load_crossfit(SD_FIG4)
gain = stats.stat_crossfit_gain(cf)
table = pandas.concat({"main stream": pandas.DataFrame(gain["main"]).T,
                       "region stream": pandas.DataFrame(gain["region"]).T})
print(table.to_string())
print("intestinal-region median range (%):", gain["region_range"])
print("cohorts with a stool stratum:", gain["n_stool_cohorts"])

                                  n_acquisitions  n_participants     median     ci_low    ci_high  n_fl_gt_mg  n_fl_lt_mg  n_nan
main stream   PREDICT                      296.0            97.0  -3.538471  -4.795255  -1.374591       106.0       190.0    0.0
              CAPSCAN_all                  282.0            15.0  18.587071  14.613301  25.840182       222.0        60.0    0.0
              CAPSCAN_stool                 56.0            15.0  -5.505770  -8.359532  -2.261450        16.0        40.0    0.0
              CAPSCAN_intestinal           226.0            14.0  28.286412  20.631113  35.260385       206.0        20.0    0.0
region stream CAPSCAN_duodenum              56.0            13.0  21.612753  12.741585  33.235931        52.0         4.0    0.0
              CAPSCAN_jejunum               65.0            14.0  35.822517  18.211445  47.480504        60.0         5.0    0.0
              CAPSCAN_ileum                 54.0            14.0  35.495740  26.991830  44.427188

## Statistic: selected-database sizes against the human reference proteome

Claim (v28d line 35): "All 726 databases generated here were smaller than the human reference proteome"; Discussion line 181: "all 726 selected databases were smaller than the human reference proteome". n = 726 acquisition-specific FASTAs (15 CAMPI, 58 healthy stool, 306 PREDICT, 347 CapScan), compared with the 20,431 entries of the reviewed human Swiss-Prot proteome. Descriptive census, no test.

In [3]:
sizes = stats.stat_database_sizes(SD_FIG1)
assert sizes["n_below_human"] == sizes["n_databases"] == 726
for k in ("n_databases", "n_below_human", "max_entries", "human_entries", "n_by_cohort"):
    print(k, sizes[k])

n_databases 726
n_below_human 726
max_entries 16923
human_entries 20431
n_by_cohort {'CapScan': 347, 'PREDICT': 306, 'HSTOOL': 58, 'CAMPI': 15}


## Statistic: reservoir sizes and computing time

Claim (v28d line 193): "study-level lookup against reservoirs containing 233–375 million sequences required 8–12 min per cohort on a single large-memory node, followed by median database-selection times of 1–3 min per acquisition and Sage searches of less than 1 min on standard CPU nodes (Supplementary Fig. 15)".

Reservoirs: entries of the four study reservoirs (healthy stool, PREDICT, CapScan, CAMPI). Lookup: wall time of the one study-level lookup task per cohort (stage `02_evidence`). Selection: for each acquisition, wall minutes of its `05_persample` and `06_parsimony` tasks summed; median over acquisitions per cohort (n = 15, 58, 306, 347). The sum of the two stage medians is printed as the alternative reading. Sage: median wall time of stage `07_sage` per cohort. Statistical unit: Slurm task; descriptive, no test.

In [4]:
times = stats.stat_compute_times(SD_SFIG15)
print("reservoir entries:", sizes["reservoir_entries"])
for k in ("lookup", "selection_per_acquisition", "selection_n", "selection_sum_of_medians", "sage"):
    print(k, times[k])

reservoir entries: {'CAMPI': 375333654, 'CapScan': 264925283, 'HSTOOL': 233497702, 'PREDICT': 272919212}
lookup {'CAMPI': 11.066666666666666, 'HSTOOL': 8.116666666666667, 'PREDICT': 12.033333333333331, 'CAPSCAN': 11.133333333333333}
selection_per_acquisition {'CAMPI': 1.2333333333333334, 'HSTOOL': 3.2083333333333335, 'PREDICT': 3.1083333333333334, 'CAPSCAN': 2.966666666666667}
selection_n {'CAMPI': 15, 'HSTOOL': 58, 'PREDICT': 306, 'CAPSCAN': 347}
selection_sum_of_medians {'CAMPI': 1.1166666666666667, 'HSTOOL': 3.1416666666666666, 'PREDICT': 3.0916666666666663, 'CAPSCAN': 2.9333333333333336}
sage {'CAMPI': 0.2333333333333333, 'HSTOOL': 0.7583333333333333, 'PREDICT': 0.9666666666666668, 'CAPSCAN': 0.6833333333333333}


## Statistic: CAMPI community members

Claim (v28d line 35): "In a defined eight-member community". n = distinct members in source_data/Fig2/member_summary.tsv.

In [5]:
n_members = stats.stat_campi_members(SD_FIG2)
print(n_members)

8


## Checks and all numbers

This unit has no quantitative figure panel, so there is no pixel check. The cell prints every recomputed value in `NUMBERS.tsv` order at full precision (`None` = OPEN).

In [6]:
print("pixel checks: none (no figure panel in this unit)")
for key, value in stats.numbers(gain, sizes, times, n_members):
    print(f"{key}\t{value!r}")

pixel checks: none (no figure panel in this unit)
L35 eight-member community	'eight'
L35 1% false discovery rate	None
L35 fewer peptides, PREDICT stool (-median gain, %)	3.5384714440068077
L35 fewer peptides, CapScan stool (-median gain, %)	5.50576989948352
L35 two stool cohorts	'two'
L35 intestinal fluid, smallest region median gain (%)	21.127414023081826
L35 intestinal fluid, largest region median gain (%)	35.82251701069244
L35 726 databases (all below human reference)	726
L181 726 databases (all below human reference)	726
L183 minimum peptide length	9
L183 maximum peptide length	50
L193 smallest reservoir (million sequences)	233.497702
L193 largest reservoir (million sequences)	375.333654
L193 shortest study-level lookup (min)	8.116666666666667
L193 longest study-level lookup (min)	12.033333333333331
L193 smallest cohort median selection time per acquisition (min)	1.2333333333333334
L193 largest cohort median selection time per acquisition (min)	3.2083333333333335
L193 largest cohor